# Week 7 · Task A — Make the four files trustworthy
**Data Visualization · Week 7: The dashboard** — Federal Transit Authority case

One notebook, top to bottom, on the **raw** files in `data/` (never edited by hand). It
1. audits all four CSVs and the GeoJSON (printed → `partA_audit.txt`),
2. repairs or drops each defect by a stated rule,
3. writes cleaned copies to `clean/`,
4. writes the decisions table → `partA_decisions.md`,
5. checks every number against the brief.

*A fix with no rule behind it is an edit, not a decision* — so every row of the decisions table has **what / how detected / rule / rows affected**, and says whether the row was **repaired** (true value can be stated) or **dropped** (it cannot).

In [1]:
import sys, os
from pathlib import Path
def _find_root():
    here = Path.cwd()
    for p in [here, here / "week7" / "assignments", here.parent / "assignments"]:
        if (p / "dashlib.py").exists() and (p / "data" / "ridership_daily.csv").exists():
            return p
    raise FileNotFoundError("Put this notebook in week7/assignments/ (next to dashlib.py) and run it from there.")
ROOT = _find_root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("working in", ROOT)

working in c:\Users\Nas Graphy\Downloads\Data-Visualization\Week 7


In [2]:
import numpy as np, pandas as pd, re
import vizlib, tslib, geolib, dashlib
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

RAW, CLEAN = Path("data"), Path("clean"); CLEAN.mkdir(exist_ok=True)

AUDIT, DEC = [], []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); AUDIT.append(s)
def section(t): log("\n" + "=" * 78 + "\n" + t + "\n" + "=" * 78)
def decide(file, what, detected, rule, rows, action):
    DEC.append(dict(file=file, what=what, detected=detected, rule=rule, rows=rows, action=action))
CHECKS = []
def check(label, got, expected, tol=0):
    ok = abs(got - expected) <= tol
    CHECKS.append((label, got, expected, ok))
    return ok

## 1 · `district_transit.csv` — the district register (25 rows)

In [3]:
section("1. district_transit.csv (the district register)")
reg_raw = pd.read_csv(RAW / "district_transit.csv", dtype={"code": str})
feats = geolib.read_geojson(RAW / "regions.geojson"); gj_codes = {f["code"] for f in feats}

log("shape:", reg_raw.shape, "| null cells:", int(reg_raw.isna().sum().sum()), "| exact duplicate rows:", int(reg_raw.duplicated().sum()))
log("dtypes:\n" + reg_raw.dtypes.to_string())
log("code unique as filed:", reg_raw.code.is_unique)
bad_codes = reg_raw.loc[~reg_raw.code.str.fullmatch(r"D\d{2}"), "code"].tolist()
log("codes not in the D## form:", bad_codes)
m_asfiled = len(gj_codes & set(reg_raw.code))
log(f"GeoJSON features matched on the register key AS FILED: {m_asfiled} of {len(gj_codes)}")
log("register codes with no polygon:", sorted(set(reg_raw.code) - gj_codes))
log(f"population / households, all {len(reg_raw)} rows: {reg_raw.population.sum():,} / {reg_raw.households.sum():,}")
reg_raw["bpr"] = reg_raw.boardings_annual / reg_raw.population
under1 = reg_raw[reg_raw.bpr < 1]
log(f"districts reading UNDER 1 boarding per resident per year: {len(under1)} ->", under1.code.tolist(),
    under1.bpr.round(4).tolist())
log("fare_concession_pct as filed, rows <= 1 (a fraction, not a %):", reg_raw.loc[reg_raw.fare_concession_pct <= 1, "code"].tolist())
log("road_km <= 0 (a code, not a length):", reg_raw.loc[reg_raw.road_km <= 0, "code"].tolist())
log("\nranges of everything we will divide by:")
log(reg_raw[["population", "households", "area_km2", "road_km", "stops"]].agg(["min", "max"]).to_string())


1. district_transit.csv (the district register)
shape: (25, 13) | null cells: 0 | exact duplicate rows: 0
dtypes:
code                       str
district                   str
population               int64
households               int64
area_km2               float64
lon                    float64
lat                    float64
income_pc_usd            int64
stops                    int64
road_km                float64
boardings_annual         int64
fare_concession_pct    float64
vehicles_registered      int64
code unique as filed: True
codes not in the D## form: ['d13', 'D 20']
GeoJSON features matched on the register key AS FILED: 22 of 24
register codes with no polygon: ['D 20', 'D00', 'd13']
population / households, all 25 rows: 60,747,000 / 9,321,212
districts reading UNDER 1 boarding per resident per year: 3 -> ['D05', 'D15', 'D23'] [0.0204, 0.0187, 0.0119]
fare_concession_pct as filed, rows <= 1 (a fraction, not a %): ['D02', 'D10', 'D18', 'D22']
road_km <= 0 (a code, not a le

In [4]:
reg = reg_raw.drop(columns="bpr").copy()

# 1a  key hygiene: "d13", "D 20"
norm = reg.code.str.replace(r"\s+", "", regex=True).str.upper()
n_key = int((norm != reg.code).sum()); reg["code"] = norm
m_after = len(gj_codes & set(reg.code))
decide("district_transit", "Key typed inconsistently ('d13', 'D 20')",
       "key regex ^D\\d{2}$ failed; GeoJSON join matched only %d/24" % m_asfiled,
       "strip whitespace, upper-case the code; keep a row only via the cleaned key", n_key, "repaired")
log(f"\n1a key repaired on {n_key} rows -> GeoJSON match {m_asfiled} -> {m_after} of 24")

# 1b  a row that is not wrong but does not belong
extra = ~reg.code.isin(gj_codes)
log("1b rows with no polygon:", reg.loc[extra, ["code", "district", "population"]].to_dict("records"))
pop_with = int(reg.population.sum())
decide("district_transit", "Extra register row D00 'Qilabad Capital Territory' (a 25th entity)",
       "no polygon for it; population adds to %s, not the federation's" % f"{pop_with:,}",
       "keep a register row only if its code has a polygon (the unit of analysis is the 24 districts)",
       int(extra.sum()), "dropped (accurate, but does not belong in a sum over districts)")
reg = reg[~extra].reset_index(drop=True)

# 1c  boardings_annual filed in thousands for 3 districts
reg["bpr"] = reg.boardings_annual / reg.population
unit_bug = reg.bpr < 1
log("1c boardings_annual rows in thousands:", reg.loc[unit_bug, "code"].tolist())
reg.loc[unit_bug, "boardings_annual"] = reg.loc[unit_bug, "boardings_annual"] * 1000
decide("district_transit", "boardings_annual filed in THOUSANDS for 3 districts (D05, D15, D23)",
       "boardings per resident per year < 1 for 3 districts; routes.csv total confirms it (see §2)",
       "if boardings/resident < 1 the value is in thousands: x 1000", int(unit_bug.sum()), "repaired")

# 1d  concession filed as a fraction
frac = reg.fare_concession_pct <= 1.0
log("1d fare_concession_pct filed as a fraction:", reg.loc[frac, "code"].tolist())
reg.loc[frac, "fare_concession_pct"] = (reg.loc[frac, "fare_concession_pct"] * 100).round(1)
decide("district_transit", "fare_concession_pct filed as a fraction (0.327) not a percent (32.7)",
       "4 values <= 1.0 in a column whose other values are 20.9-42.6",
       "a % column value <= 1 is a fraction: x 100", int(frac.sum()), "repaired")

# 1e  road_km code
rk = reg.road_km <= 0
log("1e road_km == -1 (operator code):", reg.loc[rk, "code"].tolist())
reg.loc[rk, "road_km"] = np.nan
decide("district_transit", "road_km = -1 is an operator code for 'not surveyed'",
       "impossible negative length", "set to NaN (missing is not zero); never impute",
       int(rk.sum()), "dropped the value, kept the row -> %d of %d usable" % (int(reg.road_km.notna().sum()), len(reg)))
reg = reg.drop(columns="bpr")
log("\nAFTER REPAIR: rows", len(reg), "| unique codes", reg.code.is_unique, "| GeoJSON matched", len(gj_codes & set(reg.code)))
log(f"population / households: {reg.population.sum():,} / {reg.households.sum():,}")
log(f"boardings_annual (24 districts): {reg.boardings_annual.sum():,}")
rate = reg.boardings_annual / reg.population * 1000
log(f"boardings per 1,000 residents: {rate.min():,.0f} - {rate.max():,.0f}, national pooled {reg.boardings_annual.sum() / reg.population.sum() * 1000:,.0f}")
log(f"fare_concession_pct: {reg.fare_concession_pct.min():.1f} - {reg.fare_concession_pct.max():.1f}")
log(f"road_km usable: {int(reg.road_km.notna().sum())} of {len(reg)}")

check("register rows out", len(reg), 24); check("population", int(reg.population.sum()), 57_857_000)
check("households", int(reg.households.sum()), 8_908_512); check("population if D00 kept", pop_with, 60_747_000)
check("boardings_annual after repair", int(reg.boardings_annual.sum()), 873_564_130)
check("districts under 1 boarding/resident", len(under1), 3); check("GeoJSON as filed", m_asfiled, 22); check("GeoJSON after", m_after, 24)
check("rate min per 1000", round(rate.min()), 4683); check("rate max per 1000", round(rate.max()), 30655)
check("pooled rate per 1000", round(reg.boardings_annual.sum() / reg.population.sum() * 1000), 15099)
check("concession min", round(reg.fare_concession_pct.min(), 1), 20.9); check("concession max", round(reg.fare_concession_pct.max(), 1), 42.6)
check("road_km usable", int(reg.road_km.notna().sum()), 22)


1a key repaired on 2 rows -> GeoJSON match 22 -> 24 of 24
1b rows with no polygon: [{'code': 'D00', 'district': 'Qilabad Capital Territory', 'population': 2890000}]
1c boardings_annual rows in thousands: ['D05', 'D15', 'D23']
1d fare_concession_pct filed as a fraction: ['D02', 'D10', 'D18', 'D22']
1e road_km == -1 (operator code): ['D06', 'D16']

AFTER REPAIR: rows 24 | unique codes True | GeoJSON matched 24
population / households: 57,857,000 / 8,908,512
boardings_annual (24 districts): 873,564,130
boardings per 1,000 residents: 4,683 - 30,655, national pooled 15,099
fare_concession_pct: 20.9 - 42.6
road_km usable: 22 of 24


True

## 2 · `routes.csv` — 36 routes

In [5]:
section("2. routes.csv (the route table)")
rt_raw = pd.read_csv(RAW / "routes.csv")
log("shape:", rt_raw.shape, "| route_id unique:", rt_raw.route_id.is_unique, "| null cells:", int(rt_raw.isna().sum().sum()), "| duplicate rows:", int(rt_raw.duplicated().sum()))
log("dtypes:\n" + rt_raw.dtypes.to_string())
log("\nmode value_counts AS FILED:\n" + rt_raw["mode"].value_counts().to_string())
for c in ["punctuality_pct", "cost_recovery_pct", "crowding_index", "subsidy_pkr_per_trip", "length_km", "fleet", "boardings_annual"]:
    log(f"{c:22s} min {rt_raw[c].min():>12,.2f}  max {rt_raw[c].max():>14,.2f}   values == 999: {int((rt_raw[c]==999).sum())}  == 0: {int((rt_raw[c]==0).sum())}")

rt = rt_raw.copy()
# 2a  mode labels
MODE_MAP = {"metro": "Metro", "brt": "BRT", "bus rapid transit": "BRT", "bus": "Bus", "ferry": "Ferry"}
new_mode = rt["mode"].str.strip().str.lower().map(MODE_MAP)
assert new_mode.notna().all(), "an unmapped mode label"
n_mode = int((new_mode != rt["mode"]).sum()); rt["mode"] = new_mode
log(f"\n2a mode labels: {rt_raw['mode'].nunique()} as filed -> {rt['mode'].nunique()} after; rows relabelled: {n_mode}")
decide("routes", "Mode typed 3 ways for BRT ('BRT', 'brt', 'Bus Rapid Transit') -> 6 labels for 4 modes",
       "value_counts() on mode shows 6 labels", "case-fold, then map to the 4 published modes (Metro, BRT, Bus, Ferry)", n_mode, "repaired")

# 2b  999 punctuality
code999 = rt.punctuality_pct == 999
log("2b punctuality_pct == 999:", rt.loc[code999, "route_id"].tolist())
rt.loc[code999, "punctuality_pct"] = np.nan
decide("routes", "punctuality_pct = 999.0 is an operator code (AVL feed down)",
       "a percentage above 100", "set to NaN; routes excluded from any punctuality mean, not counted as 0 or 100",
       int(code999.sum()), "dropped the value, kept the row -> %d of %d usable" % (int(rt.punctuality_pct.notna().sum()), len(rt)))

# 2c  0.0 cost recovery
zero = rt.cost_recovery_pct == 0
log("2c cost_recovery_pct == 0.0:", rt.loc[zero, ["route_id", "mode"]].to_dict("records"), "| smallest real value:", rt.loc[~zero, "cost_recovery_pct"].min())
rt.loc[zero, "cost_recovery_pct"] = np.nan
decide("routes", "cost_recovery_pct = 0.0 on a fare-charging route is a code, not a measurement",
       "exactly 0.0 on 2 routes; next-lowest real value is %.1f" % rt.loc[~zero, "cost_recovery_pct"].min(),
       "a zero where zero is impossible is a code -> NaN", int(zero.sum()), "dropped the value, kept the row")

# 2d  join to the register + the double count
parts = rt.districts_served.str.split(";")
all_codes = {c.strip() for p in parts for c in p}
misses = sorted(all_codes - set(reg.code))
n_parts = parts.str.len()
log(f"\n2d districts_served: {len(all_codes)} distinct codes; join misses against the register: {misses or 'none'}")
log(f"routes serving more than one district: {int((n_parts > 1).sum())} of {len(rt)}; mean {n_parts.mean():.2f}, max {n_parts.max()}")
ov = dashlib.overlap_factor(rt, "route_id", "districts_served", "boardings_annual")
log(f"true network boardings (each route once): {ov['true_total']:,.0f}")
log(f"attributed to every district a route serves: {ov['attributed_total']:,.0f}  = {ov['factor']:.3f}x  (exclusive: {ov['exclusive']})")
log(f"register total vs route total: {reg.boardings_annual.sum():,} vs {int(rt.boardings_annual.sum()):,}  -> differ by {abs(reg.boardings_annual.sum()-rt.boardings_annual.sum()):,} boardings")
log("routes per mode after repair:", rt["mode"].value_counts().to_dict())
log(f"punctuality_pct usable: {int(rt.punctuality_pct.notna().sum())} of {len(rt)}, range {rt.punctuality_pct.min()} - {rt.punctuality_pct.max()}")
decide("routes x register", "TWO FILES AGREE — a finding, not an error: register vs route-table boardings",
       "summed both over 12 months to 2026-08-31",
       "no change; the register total (%s) and the route total (%s) agree to within %d boardings. "
       "This confirms the x1000 repair in 1c." % (f"{reg.boardings_annual.sum():,}", f"{int(rt.boardings_annual.sum()):,}", abs(reg.boardings_annual.sum()-rt.boardings_annual.sum())),
       0, "reported")
decide("routes", "districts_served is a LIST: by-district sums from this file count a route once per district",
       "overlap_factor(): 28 of 36 routes serve >1 district",
       "never sum routes by district; the by-district map uses the register's exclusive boardings_annual. "
       "Quote %.3fx whenever a by-district route view could be added up." % ov["factor"], 0, "reported (no rows changed)")

check("routes rows", len(rt), 36); check("mode labels as filed", rt_raw["mode"].nunique(), 6); check("mode labels after", rt["mode"].nunique(), 4)
check("Bus", int((rt["mode"]=="Bus").sum()), 22); check("BRT", int((rt["mode"]=="BRT").sum()), 6); check("Metro", int((rt["mode"]=="Metro").sum()), 4); check("Ferry", int((rt["mode"]=="Ferry").sum()), 4)
check("punctuality usable", int(rt.punctuality_pct.notna().sum()), 33); check("punctuality min", rt.punctuality_pct.min(), 62.9); check("punctuality max", rt.punctuality_pct.max(), 98.8)
check("cost_recovery codes", int(zero.sum()), 2); check("route boardings total", int(rt.boardings_annual.sum()), 873_564_820)
check("attributed total", int(ov["attributed_total"]), 2_704_739_571); check("attributed factor", round(ov["factor"], 3), 3.096)
check("multi-district routes", ov["units_multi"], 28); check("mean districts per route", round(ov["mean_parts"], 2), 2.56); check("max districts", ov["max_parts"], 4)


2. routes.csv (the route table)
shape: (36, 12) | route_id unique: True | null cells: 0 | duplicate rows: 0
dtypes:
route_id                    str
mode                        str
corridor                    str
length_km               float64
stops                     int64
fleet                     int64
districts_served            str
boardings_annual          int64
punctuality_pct         float64
crowding_index          float64
cost_recovery_pct       float64
subsidy_pkr_per_trip    float64

mode value_counts AS FILED:
mode
Bus                  22
Metro                 4
Ferry                 4
BRT                   3
brt                   2
Bus Rapid Transit     1
punctuality_pct        min        62.90  max         999.00   values == 999: 3  == 0: 0
cost_recovery_pct      min         0.00  max          66.30   values == 999: 0  == 0: 2
crowding_index         min         0.40  max           1.32   values == 999: 0  == 0: 0
subsidy_pkr_per_trip   min        12.90  max          50.

True

## 3 · `ridership_daily.csv` — 2,159 rows, none of whose defects can be found by reading it
Everything here is found by *computing* something: a date parse, a duplicate count, a rule on the counts, a calendar grid.

In [6]:
section("3. ridership_daily.csv (the control-room returns)")
rd = pd.read_csv(RAW / "ridership_daily.csv", dtype={"date": str})
log("rows as filed:", len(rd), "| nulls:", int(rd.isna().sum().sum())); log("dtypes:\n" + rd.dtypes.to_string())
log("modes:", rd["mode"].value_counts().to_dict())

# 3a  two date formats in one column
s = rd.date.str.strip()
is_iso, is_dmy = s.str.fullmatch(r"\d{4}-\d{2}-\d{2}"), s.str.fullmatch(r"\d{2}/\d{2}/\d{4}")
assert (is_iso | is_dmy).all(), "a third date format"
mdy = pd.to_datetime(s[~is_iso], format="%m/%d/%Y", errors="coerce")
n_other, n_fail, n_silent = int((~is_iso).sum()), int(mdy.isna().sum()), int(mdy.notna().sum())
log(f"\n3a rows whose date is not in the main format (YYYY-MM-DD): {n_other}")
log(f"   ... that FAIL a month-first parse: {n_fail}; that a month-first parse would SILENTLY accept as a different day: {n_silent}")
date = pd.Series(pd.NaT, index=rd.index, dtype="datetime64[ns]")
date[is_iso] = pd.to_datetime(s[is_iso], format="%Y-%m-%d")
date[~is_iso] = pd.to_datetime(s[~is_iso], format="%d/%m/%Y")        # day-first, declared, never inferred
assert date.notna().all()
rd["date"] = date
log("   explicit formats used: %Y-%m-%d and %d/%m/%Y; span", rd.date.min().date(), "->", rd.date.max().date())
decide("ridership_daily", "date column holds two formats: ISO and DD/MM/YYYY",
       "385 rows fail ^YYYY-MM-DD$; a month-first parse fails on 230 and silently MIS-DATES 155",
       "parse each pattern with an explicit format (day-first for the slash dates); never let pandas infer",
       n_other, "repaired (the day can be stated: day > 12 proves day-first for 230 rows, and the pattern is constant for the 155)")

# 3b  exact re-submissions (only visible AFTER parsing the dates)
cols = ["date", "mode", "trips_scheduled", "trips_operated", "trips_on_time", "boardings", "delay_minutes"]
raw_str_dups = int(pd.read_csv(RAW / "ridership_daily.csv", dtype=str).duplicated().sum())
dup = rd.duplicated(subset=cols)
log(f"\n3b exact re-submissions: {int(dup.sum())} (a string-level duplicated() finds only {raw_str_dups}: the rest hide behind the two date formats)")
log("   by mode:", rd[dup]["mode"].value_counts().to_dict(), "| by month:", rd[dup].date.dt.to_period("M").astype(str).value_counts().to_dict())
log("   conflicting duplicates (same date+mode, different numbers):", int(rd[~dup].duplicated(["date", "mode"]).sum()))
rd_w_dups = rd.copy()
rd = rd[~dup].reset_index(drop=True)
log("   rows after de-duplication:", len(rd))
decide("ridership_daily", "30 exact re-submissions (same date, mode and every number) — all Bus, all May 2026",
       "duplicated() on all columns AFTER parsing dates (string-level finds only %d)" % raw_str_dups,
       "keep the first filing, drop repeats; there are 0 conflicting duplicates, so no judgement call about which is right",
       int(dup.sum()), "dropped (true value is the first filing; the repeat adds nothing)")

# 3c  rows that describe something that cannot happen
r1 = rd.trips_operated > rd.trips_scheduled
r2 = rd.trips_on_time > rd.trips_operated
r3 = (rd[["trips_scheduled", "trips_operated", "trips_on_time", "boardings", "delay_minutes"]] < 0).any(axis=1)
imposs = r1 | r2 | r3
log(f"\n3c impossible rows: {int(imposs.sum())}  (operated > scheduled: {int(r1.sum())}; on_time > operated: {int(r2.sum())}; negatives: {int(r3.sum())})")
log(rd[imposs].to_string())
withheld = rd[imposs].copy()
n_bd_withheld = int(withheld.boardings.sum())

# 3d  the calendar: what nobody ever filed
modes = sorted(rd["mode"].unique())
grid = pd.MultiIndex.from_product([pd.date_range("2025-04-01", "2026-09-30"), modes], names=["date", "mode"])
have = pd.MultiIndex.from_frame(rd[["date", "mode"]])
missing = grid.difference(have)
log(f"\n3d span {rd.date.min().date()} -> {rd.date.max().date()} = {len(grid)//4} days x 4 modes = {len(grid):,} slots; filed {len(have):,}; NEVER FILED {len(missing)}")
miss = pd.DataFrame(list(missing), columns=["date", "mode"]).sort_values(["mode", "date"])
log("   not filed, by mode:", miss["mode"].value_counts().to_dict())
runs = []
for md_, g in miss.groupby("mode"):
    g = g.sort_values("date"); rid = (g.date.diff().dt.days != 1).cumsum()
    for _, h in g.groupby(rid): runs.append((md_, h.date.min().date(), h.date.max().date(), len(h)))
runs = sorted(runs, key=lambda x: -x[3])
log("   longest runs of unfiled days:", [(m, str(a), str(b), n) for m, a, b, n in runs[:4]])
late = rd[rd.date >= "2026-09-22"]
log(f"   slots filed from 2026-09-22 onward: {len(late)} of 36 ->", late["mode"].value_counts().to_dict(), "(Bus and Ferry stopped filing)")
decide("ridership_daily", "63 mode-days nobody filed — not spread evenly (Ferry 22, Bus 19, Metro 18, BRT 4); longest gap 14 days (Metro, 2026-02-09 -> 2026-02-22)",
       "reindexed to the full 548-day x 4-mode calendar and counted the holes",
       "a missing day is NOT a zero: rows are never invented; series are reindexed to the calendar so the line BREAKS; "
       "rates are computed from summed counts over days that were filed; levels are shown per FILED day, never summed across a hole",
       len(missing), "not repaired (cannot be; reported)")
decide("ridership_daily", "two modes stopped filing on 2026-09-22 (Bus, Ferry): 2026-09 is a PARTIAL final month; only 18 of 36 mode-days filed after that date",
       "slots filed from 2026-09-22: 18 of 36",
       "complete months = 2025-04 -> 2026-08; 2026-09 is kept in clean/ but flagged partial_period=True and is never drawn at full weight nor used in a comparison",
       18, "flagged, not dropped")

n_imposs = int(imposs.sum())
decide("ridership_daily", "7 rows that describe something that cannot happen (trips_on_time > trips_operated)",
       "rule check on the three count columns: on_time <= operated <= scheduled, all >= 0",
       "drop the whole row: we cannot say WHICH of the three counts is wrong, so the true value cannot be stated; "
       "their %s boardings are not used (they would lift the 12-month figure by %.2f%%)" % (f"{n_bd_withheld:,}", 100*n_bd_withheld/845_531_940),
       n_imposs, "dropped (cannot be repaired); the slots count as 'withheld', so 70 mode-days have no usable row (63 + 7)")
rd_all_filed = rd.copy()                    # for the reconciliation with the register
rd = rd[~imposs].reset_index(drop=True)
rd["partial_period"] = rd.date >= "2026-09-01"

# 3e  the reconciliation (a finding)
w = lambda d: d[(d.date >= "2025-09-01") & (d.date <= "2026-08-31")]
b12 = int(w(rd_all_filed).boardings.sum()); b12_dups = int(w(rd_w_dups).boardings.sum()); b12_clean = int(w(rd).boardings.sum())
log(f"\n3e boardings, 12 months to 2026-08-31: {b12:,} (de-duplicated) | with re-submissions left in: {b12_dups:,} (+{(b12_dups/b12-1)*100:.2f}%) | after dropping the 7 impossible rows: {b12_clean:,}")
may = lambda d: d[(d.date >= "2026-05-01") & (d.date <= "2026-05-31") & (d["mode"] == "Bus")].boardings.sum()
log(f"   Bus, May 2026, with / without the re-submissions: {may(rd_w_dups):,} / {may(rd_all_filed):,} = {may(rd_w_dups)/may(rd_all_filed):.2f}x")
log(f"   register annual total vs what the returns account for: {reg.boardings_annual.sum():,} vs {b12:,}  -> the returns cover {b12/reg.boardings_annual.sum()*100:.1f}%; the missing {100-b12/reg.boardings_annual.sum()*100:.1f}% is the days nobody filed")
decide("register x ridership_daily", "TWO FILES DISAGREE — a finding, not an error: register 873,564,130 vs returns 845,531,940 boardings (12 months to 2026-08-31)",
       "summed both over the same 12 months",
       "no smoothing: the returns cover %.1f%% of the register; the %.1f%% gap is the unfiled days (+ the register being an annual attribution). "
       "Any total built from the returns is a LOWER BOUND." % (b12/reg.boardings_annual.sum()*100, 100-b12/reg.boardings_annual.sum()*100), 0, "reported")

check("daily rows as filed", len(rd_w_dups), 2159); check("non-ISO dates", n_other, 385); check("fail month-first", n_fail, 230); check("silently mis-dated", n_silent, 155)
check("re-submissions", int(dup.sum()), 30); check("rows after de-dup", len(rd_all_filed), 2129); check("impossible rows", n_imposs, 7)
check("slots in span", len(grid), 2192); check("slots never filed", len(missing), 63); check("longest gap (days)", runs[0][3], 14)
check("filed from 2026-09-22", len(late), 18); check("12m boardings", b12, 845_531_940); check("12m with dups", b12_dups, 864_127_819)
check("+% with dups (x100)", round((b12_dups/b12-1)*100, 2), 2.20, tol=0.005); check("May x", round(may(rd_w_dups)/may(rd_all_filed), 2), 2.00)
check("returns cover % of register", round(b12/reg.boardings_annual.sum()*100, 1), 96.8)


3. ridership_daily.csv (the control-room returns)
rows as filed: 2159 | nulls: 0
dtypes:
date                 str
mode                 str
trips_scheduled    int64
trips_operated     int64
trips_on_time      int64
boardings          int64
delay_minutes      int64
modes: {'Bus': 559, 'BRT': 544, 'Metro': 530, 'Ferry': 526}

3a rows whose date is not in the main format (YYYY-MM-DD): 385
   ... that FAIL a month-first parse: 230; that a month-first parse would SILENTLY accept as a different day: 155
   explicit formats used: %Y-%m-%d and %d/%m/%Y; span 2025-04-01 -> 2026-09-30

3b exact re-submissions: 30 (a string-level duplicated() finds only 23: the rest hide behind the two date formats)
   by mode: {'Bus': 30} | by month: {'2026-05': 30}
   conflicting duplicates (same date+mode, different numbers): 0
   rows after de-duplication: 2129

3c impossible rows: 7  (operated > scheduled: 0; on_time > operated: 7; negatives: 0)
           date   mode  trips_scheduled  trips_operated  trips_

np.True_

## 4 · `fare_revenue.csv` — the ministry's published table (288 rows)

In [7]:
section("4. fare_revenue.csv (the published fare table)")
fr_raw = pd.read_csv(RAW / "fare_revenue.csv")
log("shape:", fr_raw.shape, "| nulls:", int(fr_raw.isna().sum().sum()), "| duplicate (month, mode, ticket_type):", int(fr_raw.duplicated(["month", "mode", "ticket_type"]).sum()))
log("dtypes:\n" + fr_raw.dtypes.to_string()); log("ticket_type:", fr_raw.ticket_type.value_counts().to_dict()); log("months:", fr_raw.month.min(), "->", fr_raw.month.max(), f"({fr_raw.month.nunique()})")
log("min trips / revenue:", fr_raw.trips.min(), fr_raw.revenue_pkr.min())

fr = fr_raw.copy()
fr["is_total"] = fr.ticket_type.str.strip() == "All tickets"
parts_ = fr[~fr.is_total]
log(f"\nrows {len(fr)} = {len(parts_)} parts + {int(fr.is_total.sum())} that are not parts")
for v in ["trips", "revenue_pkr"]:
    dd = dashlib.drilldown_check(fr, "ticket_type", v, within=("month", "mode"), total_label="All tickets")
    log(f"drilldown_check({v}): summing EVERY row = {dd['sum_all_pct']:.1f}% of the parts | reconciles: {dd['reconciles']} | worst disagreement {dd['worst_abs']:.0f}")
    check(f"sum-every-row % {v}", round(dd["sum_all_pct"], 1), 200.0); assert dd["reconciles"]
fr["complete_month"] = fr.month <= "2026-08"
P = fr[~fr.is_total]
tot_trips, tot_rev = int(P.trips.sum()), int(P.revenue_pkr.sum())
log(f"journeys / revenue over the 18 months: {tot_trips:,} / {tot_rev:,} PKR | mean fare {tot_rev/tot_trips:.1f} PKR per journey")
dec_ = dashlib.drilldown_check(fr, "ticket_type", "trips", within=("month", "mode"), total_label="All tickets")
decide("fare_revenue", "72 'All tickets' rows are the publisher's own subtotal filed in the same column as the parts",
       "drilldown_check(): summing every row gives exactly 200.0% of the journeys and of the revenue",
       "flag is_total=True and exclude from every sum; use them only as a reconciliation check — the parts agree with them EXACTLY in every month for every mode",
       int(fr.is_total.sum()), "kept in the file, flagged; excluded from sums (accurate, but not parts)")
decide("fare_revenue", "2026-09 is a partial final month (control rooms of Bus and Ferry stopped filing on 22 Sep)",
       "daily returns end early for 2 modes; the table has 18 months and the last cannot be shown to be whole",
       "complete_month=False for 2026-09; all comparisons use 2025-04 -> 2026-08", int((~fr.complete_month).sum()), "flagged, not dropped")

# 4b  boardings are not journeys
b18 = rd[(rd.date >= "2025-04-01") & (rd.date <= "2026-08-31")].boardings.sum()
j18 = P[P.complete_month].trips.sum()
ratio = b18 / j18
log(f"boardings / journeys, complete months, 7 impossible rows dropped: {ratio:.3f} (every 1,000 journeys produce about {ratio*1000:,.0f} boardings: people change vehicles)")
decide("ridership_daily x fare_revenue", "boardings (legs) and journeys (transfer-linked trips) are different units",
       "boardings / journeys over the complete months = %.3f" % ratio,
       "never add them, never put them on one axis; every figure names its unit in words", 0, "reported (no rows changed)")

check("fare rows", len(fr), 288); check("parts", len(parts_), 216); check("totals", int(fr.is_total.sum()), 72)
check("18-month journeys", tot_trips, 934_051_014); check("18-month revenue", tot_rev, 36_669_953_119); check("mean fare", round(tot_rev/tot_trips, 1), 39.3)
check("boardings/journeys ~1.30", round(ratio, 2), 1.30, tol=0.006)


4. fare_revenue.csv (the published fare table)
shape: (288, 5) | nulls: 0 | duplicate (month, mode, ticket_type): 0
dtypes:
month            str
mode             str
ticket_type      str
trips          int64
revenue_pkr    int64
ticket_type: {'All tickets': 72, 'Daily': 72, 'Monthly': 72, 'Single': 72}
months: 2025-04 -> 2026-09 (18)
min trips / revenue: 56946 2271614

rows 288 = 216 parts + 72 that are not parts
drilldown_check(trips): summing EVERY row = 200.0% of the parts | reconciles: True | worst disagreement 0
drilldown_check(revenue_pkr): summing EVERY row = 200.0% of the parts | reconciles: True | worst disagreement 0
journeys / revenue over the 18 months: 934,051,014 / 36,669,953,119 PKR | mean fare 39.3 PKR per journey
boardings / journeys, complete months, 7 impossible rows dropped: 1.303 (every 1,000 journeys produce about 1,303 boardings: people change vehicles)


np.True_

## 5 · Write the cleaned files — by this script, never by hand

In [8]:
# the full calendar, with a status for every mode-day (so later notebooks can draw gaps as gaps)
status = pd.DataFrame(list(grid), columns=["date", "mode"])
filed_idx, with_idx = set(zip(rd_all_filed.date, rd_all_filed["mode"])), set(zip(withheld.date, withheld["mode"]))
status["status"] = [("withheld_impossible" if k in with_idx else "filed" if k in filed_idx else "not_filed") for k in zip(status.date, status["mode"])]
grid_df = status.merge(rd, on=["date", "mode"], how="left").drop(columns="partial_period")
grid_df["partial_period"] = grid_df.date >= "2026-09-01"
log("\ncalendar status counts:", status.status.value_counts().to_dict())

reg.to_csv(CLEAN / "district_transit_clean.csv", index=False)
rt.to_csv(CLEAN / "routes_clean.csv", index=False)
rd.assign(date=rd.date.dt.strftime("%Y-%m-%d")).to_csv(CLEAN / "ridership_daily_clean.csv", index=False)
grid_df.assign(date=grid_df.date.dt.strftime("%Y-%m-%d")).to_csv(CLEAN / "ridership_calendar.csv", index=False)
fr.to_csv(CLEAN / "fare_revenue_clean.csv", index=False)
log("wrote:", sorted(p.name for p in CLEAN.glob("*.csv")))

# decisions table -> markdown
def esc(x): return str(x).replace("|", "\\|").replace("\n", " ")
lines = ["# Task A — decisions table", "",
         "One row per decision. *Repaired* = the true value can be stated and why; *dropped* = it cannot.", "",
         "| # | File | What | How detected | Rule applied | Rows affected | Action |", "|---|---|---|---|---|---|---|"]
for i, d in enumerate(DEC, 1):
    lines.append(f"| {i} | {d['file']} | {esc(d['what'])} | {esc(d['detected'])} | {esc(d['rule'])} | {d['rows']} | {esc(d['action'])} |")
Path("partA_decisions.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"{len(DEC)} decisions written to partA_decisions.md")

section("6. verification against the brief")
for lab, got, exp, ok in CHECKS: log(f"  {'OK ' if ok else 'XX '} {lab:36s} got {got!s:>16}  expected {exp!s:>16}")
log(f"\n{sum(c[3] for c in CHECKS)} of {len(CHECKS)} checks match the brief")
Path("partA_audit.txt").write_text("\n".join(AUDIT) + "\n", encoding="utf-8")
assert all(c[3] for c in CHECKS), "a check disagrees with the brief"


calendar status counts: {'filed': 2122, 'not_filed': 63, 'withheld_impossible': 7}
wrote: ['district_transit_clean.csv', 'fare_revenue_clean.csv', 'ridership_calendar.csv', 'ridership_daily_clean.csv', 'routes_clean.csv']
19 decisions written to partA_decisions.md

6. verification against the brief
  OK  register rows out                    got               24  expected               24
  OK  population                           got         57857000  expected         57857000
  OK  households                           got          8908512  expected          8908512
  OK  population if D00 kept               got         60747000  expected         60747000
  OK  boardings_annual after repair        got        873564130  expected        873564130
  OK  districts under 1 boarding/resident  got                3  expected                3
  OK  GeoJSON as filed                     got               22  expected               22
  OK  GeoJSON after                        got               2

In [9]:
import pandas as pd
pd.DataFrame(DEC).rename(columns={"file":"File","what":"What","detected":"How detected","rule":"Rule applied","rows":"Rows","action":"Action"})

,File,What,How detected,Rule applied,Rows,Action
0,district_transit,"Key typed inconsistently ('d13', 'D 20')",key regex ^D\d{2}$ failed; GeoJSON join matche...,"strip whitespace, upper-case the code; keep a ...",2,repaired
1,district_transit,Extra register row D00 'Qilabad Capital Territ...,"no polygon for it; population adds to 60,747,0...",keep a register row only if its code has a pol...,1,"dropped (accurate, but does not belong in a su..."
2,district_transit,boardings_annual filed in THOUSANDS for 3 dist...,boardings per resident per year < 1 for 3 dist...,if boardings/resident < 1 the value is in thou...,3,repaired
3,district_transit,fare_concession_pct filed as a fraction (0.327...,4 values <= 1.0 in a column whose other values...,a % column value <= 1 is a fraction: x 100,4,repaired
4,district_transit,road_km = -1 is an operator code for 'not surv...,impossible negative length,set to NaN (missing is not zero); never impute,2,"dropped the value, kept the row -> 22 of 24 us..."
5,routes,"Mode typed 3 ways for BRT ('BRT', 'brt', 'Bus ...",value_counts() on mode shows 6 labels,"case-fold, then map to the 4 published modes (...",3,repaired
6,routes,punctuality_pct = 999.0 is an operator code (A...,a percentage above 100,set to NaN; routes excluded from any punctuali...,3,"dropped the value, kept the row -> 33 of 36 us..."
7,routes,cost_recovery_pct = 0.0 on a fare-charging rou...,exactly 0.0 on 2 routes; next-lowest real valu...,a zero where zero is impossible is a code -> NaN,2,"dropped the value, kept the row"
8,routes x register,"TWO FILES AGREE — a finding, not an error: reg...",summed both over 12 months to 2026-08-31,"no change; the register total (873,564,130) an...",0,reported
9,routes,districts_served is a LIST: by-district sums f...,overlap_factor(): 28 of 36 routes serve >1 dis...,never sum routes by district; the by-district ...,0,reported (no rows changed)
